In [1]:
import json
import os
from pathlib import Path

In [2]:
evidence_path = Path('../reports/evidence/evidence_packets.json')

with open(evidence_path) as f:
    evidence_packets = json.load(f)

evidence_packets.keys()

dict_keys(['TP', 'FN', 'FP', 'TN'])

In [3]:
tp_packet = evidence_packets['TP']

print(json.dumps(tp_packet, indent=2))

{
  "patient_label": "TP_case",
  "test_row_index": 14271,
  "prediction": {
    "y_true": 1,
    "y_pred": 1,
    "y_proba": 0.999529083549892,
    "prediction_type": "TP"
  },
  "risk_increasing_evidence": [
    {
      "feature": "d1_heartrate_min",
      "value": 0.0,
      "shap_value": 2.0046439410543497,
      "direction": "risk_increasing",
      "clinical_meaning": "extreme or very low heart rate may indicate severe instability or data quality issue",
      "caution_flags": [
        "Zero-valued vital sign; may reflect extreme clinical event or recording artifact."
      ]
    },
    {
      "feature": "d1_spo2_min",
      "value": 31.0,
      "shap_value": 0.9487559313200912,
      "direction": "risk_increasing",
      "clinical_meaning": "low minimum oxygen saturation indicates hypoxemia",
      "caution_flags": []
    },
    {
      "feature": "ventilated_apache",
      "value": 1.0,
      "shap_value": 0.7250477735108737,
      "direction": "risk_increasing",
      "clini

In [6]:
def build_explanation_prompt(evidence_packet):
    prompt = f"""
You are a clinical AI explanation assistant.

Your task is to explain why the mortality prediction model made its prediction for this ICU patient.

Use only the provided evidence. Do not invent clinical facts. If a feature has a caution flag, mention it carefully.

Patient prediction:
- Predicted label: {evidence_packet['prediction']['y_pred']}
- Predicted mortality probability: {evidence_packet['prediction']['y_proba']:.4f}

Risk-increasing evidence:
{json.dumps(evidence_packet['risk_increasing_evidence'], indent=2)}

Risk-decreasing evidence:
{json.dumps(evidence_packet['risk_decreasing_evidence'], indent=2)}

Write the explanation with the following structure:

1. Prediction summary
2. Main risk-increasing factors
3. Main risk-decreasing factors
4. Caution notes
5. Overall interpretation

Keep the explanation clinically grounded and concise.
"""
    return prompt.strip()

In [7]:
tp_prompt = build_explanation_prompt(evidence_packets['TP'])

print(tp_prompt)

You are a clinical AI explanation assistant.

Your task is to explain why the mortality prediction model made its prediction for this ICU patient.

Use only the provided evidence. Do not invent clinical facts. Do not use the true label to justify the prediction; use it only as case metadata. If a feature has a caution flag, mention it carefully.

Patient prediction:
- Case type: TP
- True label: 1
- Predicted label: 1
- Predicted mortality probability: 0.9995

Risk-increasing evidence:
[
  {
    "feature": "d1_heartrate_min",
    "value": 0.0,
    "shap_value": 2.0046439410543497,
    "direction": "risk_increasing",
    "clinical_meaning": "extreme or very low heart rate may indicate severe instability or data quality issue",
    "caution_flags": [
      "Zero-valued vital sign; may reflect extreme clinical event or recording artifact."
    ]
  },
  {
    "feature": "d1_spo2_min",
    "value": 31.0,
    "shap_value": 0.9487559313200912,
    "direction": "risk_increasing",
    "clinical

In [9]:
tp_manual_explanation = """
1. Prediction summary
The model assigned this patient a very high predicted probability of in-hospital mortality (0.9995).

2. Main risk-increasing factors
The prediction was mainly driven upward by severe physiological instability. The strongest risk-increasing evidence was d1_heartrate_min = 0, which had the largest positive SHAP contribution. This value should be interpreted cautiously because a zero-valued heart rate may represent either an extreme clinical event or a recording artifact. Other major risk-increasing factors included very low minimum oxygen saturation (d1_spo2_min = 31), mechanical ventilation, very low systolic blood pressure (d1_sysbp_min = 41), low GCS motor score, low bicarbonate values, and low platelet count.

3. Main risk-decreasing factors
A few features slightly decreased the predicted risk, including d1_wbc_min, d1_bun_max, hematocrit-related features, BMI, and height. However, these negative contributions were much smaller than the strong risk-increasing signals.

4. Caution notes
The main caution flag is d1_heartrate_min = 0. This may reflect a true extreme clinical event, but it may also represent a data quality issue or recording artifact. Therefore, this feature should be interpreted carefully.

5. Overall interpretation
Overall, the model's high-risk prediction is clinically understandable based on the provided evidence. The patient had multiple strong risk-increasing signals related to hypoxemia, mechanical ventilation, hypotension, impaired neurological response, possible metabolic acidosis, and thrombocytopenia. These factors collectively pushed the prediction strongly toward mortality risk.
"""
print(tp_manual_explanation)


1. Prediction summary
The model assigned this patient a very high predicted probability of in-hospital mortality (0.9995).

2. Main risk-increasing factors
The prediction was mainly driven upward by severe physiological instability. The strongest risk-increasing evidence was d1_heartrate_min = 0, which had the largest positive SHAP contribution. This value should be interpreted cautiously because a zero-valued heart rate may represent either an extreme clinical event or a recording artifact. Other major risk-increasing factors included very low minimum oxygen saturation (d1_spo2_min = 31), mechanical ventilation, very low systolic blood pressure (d1_sysbp_min = 41), low GCS motor score, low bicarbonate values, and low platelet count.

3. Main risk-decreasing factors
A few features slightly decreased the predicted risk, including d1_wbc_min, d1_bun_max, hematocrit-related features, BMI, and height. However, these negative contributions were much smaller than the strong risk-increasing 

In [10]:
prompts = {
    case_type: build_explanation_prompt(packet)
    for case_type, packet in evidence_packets.items()
}

prompts.keys()

dict_keys(['TP', 'FN', 'FP', 'TN'])

In [11]:
os.makedirs('../reports/llm_reasoning/prompts', exist_ok=True)

for case_type, prompt in prompts.items():
    with open(f'../reports/llm_reasoning/prompts/{case_type.lower()}_prompt.txt', 'w') as f:
        f.write(prompt)

print('Prompts saved.')

Prompts saved.


In [12]:
print(prompts['FN'])

You are a clinical AI explanation assistant.

Your task is to explain why the mortality prediction model made its prediction for this ICU patient.

Use only the provided evidence. Do not invent clinical facts. Do not use the true label to justify the prediction; use it only as case metadata. If a feature has a caution flag, mention it carefully.

Patient prediction:
- Case type: FN
- True label: 1
- Predicted label: 0
- Predicted mortality probability: 0.0037

Risk-increasing evidence:
[
  {
    "feature": "age",
    "value": 72.0,
    "shap_value": 0.3049919535072186,
    "direction": "risk_increasing",
    "clinical_meaning": "older age is associated with higher mortality risk",
    "caution_flags": []
  },
  {
    "feature": "d1_resprate_max",
    "value": 56.0,
    "shap_value": 0.1828680414016568,
    "direction": "risk_increasing",
    "clinical_meaning": "high respiratory rate may indicate respiratory distress",
    "caution_flags": []
  },
  {
    "feature": "apache_3j_diagnosi

In [13]:
os.makedirs('../reports/llm_reasoning/manual_explanations', exist_ok=True)

with open('../reports/llm_reasoning/manual_explanations/tp_manual_explanation.txt', 'w') as f:
    f.write(tp_manual_explanation)

print('Manual explanation saved.')

Manual explanation saved.
